In [13]:
import re
import yaml
import pandas as pd
from pathlib import Path

ACCESS_PATTERNS = [
    (r'^nodes_walk_', 'nodes_walk'),
    (r'^zones_transit_jobs_', 'transit_jobs'),
    (r'jobs_within_\d+_min', 'travel_time_jobs'),
    (r'^zones_', 'zones_any'),
    (r'logsum_', 'logsum'),
    (r'^parcels_zones_', 'parcels_zones_bridge'),
]

def classify_accessibility(var_name):
    for pat, cat in ACCESS_PATTERNS:
        if re.search(pat, var_name):
            return True, cat
    return False, None

def extract_building_type_id(file_name):
    match = re.search(r'(\d{2})(?:\.\w+)?$', file_name)
    return int(match.group(1)) if match else None

def load_coefficients_from_yaml(yaml_path):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
    return dict(data.get('fit_parameters', {}).get('Coefficient', {}))

def summarize_yaml_dir_by_building_type(yaml_dir):
    yaml_dir = Path(yaml_dir)
    records = []

    for path in yaml_dir.glob("*.y*ml"):
        coefs = load_coefficients_from_yaml(path)
        bldg_id = extract_building_type_id(path.stem)
        for var, beta in coefs.items():
            if var == "Intercept":
                continue
            is_acc, cat = classify_accessibility(var)
            records.append({
                "building_type_id": bldg_id,
                "model": path.stem,
                "variable": var,
                "beta": float(beta),
                "abs_beta": abs(float(beta)),
                "is_access": is_acc,
                "access_category": cat,
            })

    df = pd.DataFrame(records)
    if df.empty:
        print("No YAML files found or coefficients missing.")
        return None, None, None, None

    # ---- Per building type summary ----
    bsum = (
        df.groupby("building_type_id")
        .agg(total_vars=("variable", "count"), access_vars=("is_access", "sum"))
        .reset_index()
    )
    bsum["access_pct"] = bsum["access_vars"] / bsum["total_vars"] * 100
    top5 = bsum.sort_values("access_pct", ascending=False).head(5)

    # ---- Top 10 most frequent accessibility variables ----
    top_freq = (
        df[df["is_access"]]
        .groupby("variable")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(10)
    )

    # ---- Top 10 by highest absolute coefficient overall ----
    top_coef = (
        df[df["is_access"]]
        .sort_values("abs_beta", ascending=False)
        .head(10)[["variable", "beta", "abs_beta", "model", "building_type_id"]]
    )

    # ---- NEW: Average absolute coefficient across all models ----
    top_overall_abs = (
        df[df["is_access"]]
        .groupby("variable")
        .agg(mean_abs_coef=("abs_beta", "mean"), count=("variable", "count"))
        .sort_values("mean_abs_coef", ascending=False)
        .head(10)
        .reset_index()
    )

    # --- Print summaries ---
    print("\n=== Top 5 building types with highest accessibility variable share ===")
    print(top5.to_string(index=False))

    print("\n=== Top 10 most common accessibility variables across all building types ===")
    print(top_freq.to_string(index=False))

    print("\n=== Top 10 accessibility variables with highest single-model absolute coefficients ===")
    print(top_coef.to_string(index=False))

    print("\n=== Top 10 accessibility variables by average absolute coefficient across all models ===")
    print(top_overall_abs.to_string(index=False))

    print("\n⚠️  Reminder: Coefficient magnitude reflects sensitivity *given variable scale*.")
    print("   If variables weren’t standardized, interpret these as relative influence, not direct comparability.\n")

    return bsum, top_freq, top_coef, top_overall_abs


In [14]:
bsum, top_freq, top_coef, top_overall_abs = summarize_yaml_dir_by_building_type("/mnt/semcog_urbansim/configs/repm_2050")


=== Top 5 building types with highest accessibility variable share ===
 building_type_id  total_vars  access_vars  access_pct
               91          38           31   81.578947
               42          63           51   80.952381
               94          62           50   80.645161
               84          56           45   80.357143
               93          35           28   80.000000

=== Top 10 most common accessibility variables across all building types ===
                    variable  count
 zones_logsum_pop_mid_income     89
 zones_logsum_pop_low_income     89
  nodes_walk_node_r1500_sqft     89
zones_logsum_job_high_income     89
 zones_logsum_job_low_income     89
     nodes_walk_housing_cost     89
 zones_logsum_job_mid_income     89
zones_logsum_pop_high_income     89
                 zones_acres     87
          jobs_within_30_min     87

=== Top 10 accessibility variables with highest single-model absolute coefficients ===
                       variable     